In [ ]:
# open one mode and run and try combining for wind shear variables
from pathlib import Path
import xarray as xr

# source folder
folder = Path(
    "/global/cfs/projectdirs/m2637/ALCC/"
    "ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001"
)

variables = ["U200", "U850", "V200", "V850", "lat", "lon"]

# find all NetCDF files
files = sorted(folder.glob("*.nc"))

# combine files, retaining only the four variables
ds = xr.open_mfdataset(
    files,
    combine="by_coords",
    preprocess=lambda d: d[variables],
    parallel=False,
    chunks={},  # Use the stored chunk layout
)

# # check
# print(ds["U200"].data.chunksize)
# print(ds["U200"].data.chunks)

In [ ]:
# clip to NAtl region
import numpy as np

# Load only the small, fixed coordinate arrays into memory
lat = ds["lat"].isel(time=0).values
lon = ds["lon"].isel(time=0).values

# Convert longitude from 0–360 to -180–180
lon = ((lon + 180) % 360) - 180

# North Atlantic: 0–90°N, 120°W–25°E
region_mask = (
    (lat >= 0) &
    (lat <= 90) &
    (lon >= -120) &
    (lon <= 25)
)

# Indices of grid cells inside the region
region_indices = np.flatnonzero(region_mask)

print(f"Regional grid cells: {len(region_indices):,}")
print(f"Total grid cells: {len(lat):,}")
print(f"Fraction retained: {len(region_indices) / len(lat):.1%}")

In [ ]:
# Filter to hurricane season
seasonal = ds.isel(
    time=np.flatnonzero(
        ds.time.dt.month.isin([6, 7, 8, 9, 10]).values
    )
)

# Select the North Atlantic grid cells
regional = seasonal.isel(ncol=region_indices)

# Now rechunk the smaller regional dataset
regional = regional.chunk({
    "time": 7,
    "ncol": 5000,
})

print("U200 chunks after filtering and rechunking:")
print(regional["U200"].data.chunksize)

In [ ]:
# Get regional coordinates
lat_reg = lat[region_indices]
lon_reg = lon[region_indices]

# Assign each cell to the southwest corner of its 5-degree bin
lat_bin = np.floor(lat_reg / 5) * 5
lon_bin = np.floor(lon_reg / 5) * 5

# Attach bin labels to the regional dataset
regional = regional.assign_coords(
    lat_bin=("ncol", lat_bin),
    lon_bin=("ncol", lon_bin),
)

# Inspect the number of unique bins
bin_pairs = np.column_stack([lat_bin, lon_bin])
unique_bins = np.unique(bin_pairs, axis=0)

print(f"Number of occupied 5° × 5° bins: {len(unique_bins):,}")
print("Example bin coordinates:")
print(unique_bins[:10])

print(f"Hurricane-season time steps: {regional.sizes['time']:,}")

In [ ]:
import dask
import gc

# Create one integer ID for each 5-degree bin
bin_pairs = np.column_stack([lat_bin, lon_bin])
unique_bins, bin_id = np.unique(
    bin_pairs, axis=0, return_inverse=True
)

regional = regional.assign_coords(
    bin_id=("ncol", bin_id)
)

# Small accumulators for the final result
n_bins = len(unique_bins)
total_sum = np.zeros(n_bins, dtype=np.float64)
total_count = np.zeros(n_bins, dtype=np.int64)

# Process 7 time steps per batch
batch_size = 7
n_times = regional.sizes["time"]

for start in range(0, n_times, batch_size):
    stop = min(start + batch_size, n_times)
    block = regional.isel(time=slice(start, stop))

    # Calculate vector shear magnitude
    du = block["U200"] - block["U850"]
    dv = block["V200"] - block["V850"]
    shear = np.hypot(du, dv)

    # Aggregate just this batch
    block_sum = shear.groupby("bin_id").sum(
        dim=("time", "ncol"), skipna=True
    )
    block_count = shear.groupby("bin_id").count(
        dim=("time", "ncol")
    )

    # Compute the small aggregated results
    sum_result, count_result = dask.compute(
        block_sum, block_count
    )

    # Accumulate sums and counts
    total_sum += sum_result.values
    total_count += count_result.values

    if start % 700 == 0:
        print(f"Processed {stop:,} / {n_times:,} time steps")

    del block, du, dv, shear
    del block_sum, block_count, sum_result, count_result
    gc.collect()

# Calculate final means
mean_values = np.divide(
    total_sum,
    total_count,
    out=np.full(n_bins, np.nan),
    where=total_count > 0,
)

# Create the small output dataset
result = xr.Dataset({
    "mean_shear_850_200": ("bin_id", mean_values),
    "valid_count": ("bin_id", total_count),
}, coords={
    "bin_id": np.arange(n_bins),
    "lat_bin": ("bin_id", unique_bins[:, 0]),
    "lon_bin": ("bin_id", unique_bins[:, 1]),
})

print(result)

In [ ]:
# Calculate final means
mean_values = np.divide(
    total_sum,
    total_count,
    out=np.full(n_bins, np.nan),
    where=total_count > 0,
)

# Create the small output dataset
result = xr.Dataset({
    "mean_shear_850_200": ("bin_id", mean_values),
    "valid_count": ("bin_id", total_count),
}, coords={
    "bin_id": np.arange(n_bins),
    "lat_bin": ("bin_id", unique_bins[:, 0]),
    "lon_bin": ("bin_id", unique_bins[:, 1]),
})

print(result)

In [ ]:
# # CRASHED HERE
# # Sum shear across time and grid cells within each bin
# shear_sum = shear.groupby("bin_id").sum(
#     dim=("time", "ncol"),
#     skipna=True,
# )

# # Count valid observations in each bin
# shear_count = shear.groupby("bin_id").count(
#     dim=("time", "ncol"),
# )

# # Calculate the mean
# mean_shear = (
#     shear_sum / shear_count.where(shear_count > 0)
# ).rename("mean_shear_850_200")

# print(mean_shear)